## Integer Programming: 0-1 Knapsack Problem

Given a list of item values and weights and a knapsack weight capacity, determine the items to put into the knapsack to maximize the total value, without exceeding the weight.

This is of course a very well-known problem, and it can be formulated as follows:

$$
\begin{aligned}
\max\quad & \sum_{i=1}^{n} v_i x_i \\
\text{s.t.}\quad & \sum_{i=1}^{n} w_i x_i \le W \\
& x_i \in \left\{ 0,1 \right\} \quad \forall i = 1,2,\dots,n
\end{aligned}
$$

where $v_i$ and $w_i$ is the value of item $i$, $x_i$ represents whether we select item $i$ and $W$ is the weight capacity of the knapsack.

In [2]:
# Imports and input data

import numpy as np
import pandas as pd
from scipy.optimize import LinearConstraint, milp, Bounds

# Set seed for repeatable results
np.random.seed(42)

item_values = np.random.randint(2, 10, 20)
item_weights = np.random.randint(2, 10, 20)

capacity = 20

print("Values:", item_values)
print("Weights:", item_weights)

Values: [8 5 6 8 4 9 6 6 8 3 4 8 4 4 9 6 5 9 9 4]
Weights: [7 6 3 9 5 7 7 3 9 5 6 2 5 3 7 6 5 2 2 4]


In [3]:
# Set objective and constraints

items = len(item_values)

# Flip sign to maximize instead of minimize
coef = -item_values

constraint = LinearConstraint(item_weights, ub=capacity)

# Make sure each item is in {0, 1}
bounds = Bounds(np.zeros(items), np.ones(items))

In [4]:
# Run solver

# Set all variables to be integers
integrality = np.ones_like(coef)

res = milp(c=coef, integrality=integrality, bounds=bounds, constraints=constraint)
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -47.0
              x: [ 0.000e+00  0.000e+00 ...  1.000e+00  0.000e+00]
 mip_node_count: 1
 mip_dual_bound: -47.0
        mip_gap: 0.0

In [ ]:
# Display solution

df = pd.DataFrame(dict(value=item_values, weight=item_weights))
df = df[res.x > 0]
df = pd.concat([df, df.sum().to_frame("sum").T, pd.DataFrame({"value": "", "weight": 20}, index=["capacity"])])[["value", "weight"]]
df.index.name = "item"
df.columns.name = "Optimal selection"

sum_row = pd.IndexSlice[df.index[df.index == "sum"], :]
df.style.map(lambda _: "font-weight: bold", subset=sum_row)

Optimal selection,value,weight
item,,
2,6,3
7,6,3
11,8,2
14,9,7
17,9,2
18,9,2
sum,47,19
capacity,,20


## Greedy Heuristic

Another way to solve the problem would be to use the following algorithm:

- Process items by weight-to-value ratios, from large to small.
- For each item, put it in the knapsack if it fits. Otherwise skip it.

In [ ]:
# Greedy heuristic

df = pd.DataFrame(dict(value=item_values, weight=item_weights))
df["ratio"] = df.value / df.weight

# Pick items by ratio
df_out = pd.DataFrame()
for _, item in df.sort_values("ratio", ascending=False)[["value", "weight"]].iterrows():
    if df_out.empty or df_out.weight.sum() + item.weight < capacity:
        df_out = pd.concat([df_out, item.to_frame().T])

# Display solution
value_sum = df_out.value.sum()
df_out = pd.concat([
    df_out,
    df_out.sum().to_frame("sum").T,
    pd.DataFrame({"value": "", "weight": 20}, index=["capacity"]),
    pd.DataFrame({"value": int(-res.fun), "weight": ""}, index=["optimal"]),
    pd.DataFrame({"value": f"{(-res.fun - value_sum) / -res.fun:.2%}", "weight": ""}, index=["optimality gap"]),
])
df_out.index.name = "item"
df_out.columns.name = "Greedy selection"

sum_row = pd.IndexSlice[df_out.index[df_out.index == "sum"], :]
df_out.style.map(lambda _: "font-weight: bold", subset=sum_row)

Greedy selection,value,weight
item,,
18,9,2
17,9,2
11,8,2
2,6,3
7,6,3
13,4,3
19,4,4
sum,46,19
capacity,,20
